In [1]:
import pandas as pd
from pathlib import Path

PROCESSED = Path("../data/processed")

nodes = pd.read_csv(
    PROCESSED / "nodes.csv",
    usecols=["deployment_id", "platform", "in_catalogue"],
    dtype=str,
    keep_default_na=False,
)
edges = pd.read_csv(
    PROCESSED / "edges.csv",
    dtype={"source": str, "target": str},
    keep_default_na=False,
)

connected_ids = set(edges["source"]) | set(edges["target"])

print("Nodes:", len(nodes))
print("Connected nodes:", len(connected_ids))

Nodes: 86495
Connected nodes: 51218


In [2]:
import sys
sys.path.insert(0, "..")
from src.pipeline.edge_generator import load_unique_voter_dao_pairs

voter_dao_pairs = load_unique_voter_dao_pairs()
voter_dao_pairs = voter_dao_pairs.dropna(subset=["voter"])
voter_dao_pairs = voter_dao_pairs[voter_dao_pairs["voter"] != ""]

voters_per_dao = voter_dao_pairs.groupby("deployment_id").size()

print("Voter-DAO pairs:", len(voter_dao_pairs))
print("Deployments with votes:", len(voters_per_dao))

Voter-DAO pairs: 9250601
Deployments with votes: 66086


In [3]:
def classify(deployment_id):
    if deployment_id in connected_ids:
        return "connected"
    if deployment_id in voters_per_dao.index:
        return "votes_no_edges"
    return "no_votes"

nodes["status"] = nodes["deployment_id"].map(classify)
nodes["voters_in_votes"] = (
    nodes["deployment_id"].map(voters_per_dao).fillna(0).astype(int)
)

print(nodes["status"].value_counts())

pd.crosstab(nodes["platform"], nodes["status"], margins=True, margins_name="total")

status
connected         51218
no_votes          20409
votes_no_edges    14868
Name: count, dtype: int64


status,connected,no_votes,votes_no_edges,total
platform,,,,
aragon,562,1408,432,2402
daohaus,787,2521,151,3459
daostack,36,3,19,58
realms,780,2771,673,4224
snapshot,48148,10461,13349,71958
tally,905,3245,244,4394
total,51218,20409,14868,86495


In [4]:
daos_per_voter = voter_dao_pairs.groupby("voter").size()
pairs = voter_dao_pairs.assign(
    voter_daos=voter_dao_pairs["voter"].map(daos_per_voter)
)

# Voters removed by edge_generator.py (more than 100 DAOs), per deployment
filtered_out = pairs[pairs["voter_daos"] > 100].groupby("deployment_id").size()
nodes["voters_filtered_out"] = (
    nodes["deployment_id"].map(filtered_out).fillna(0).astype(int)
)

isolated = nodes[nodes["status"] == "votes_no_edges"]
print("Isolated with votes:", len(isolated))
print("  only single-DAO voters:", (isolated["voters_filtered_out"] == 0).sum())
print("  isolated because of the filter:", (isolated["voters_filtered_out"] > 0).sum())

Isolated with votes: 14868
  only single-DAO voters: 14747
  isolated because of the filter: 121


In [5]:
bins = [0, 1, 5, 20, 100, float("inf")]
labels = ["1", "2-5", "6-20", "21-100", ">100"]

with_votes = nodes[nodes["voters_in_votes"] > 0].copy()
with_votes["size_band"] = pd.cut(with_votes["voters_in_votes"], bins=bins, labels=labels)

print(with_votes.groupby("status")["voters_in_votes"].median())
pd.crosstab(with_votes["size_band"], with_votes["status"], margins=True, margins_name="total")

status
connected         1.0
votes_no_edges    1.0
Name: voters_in_votes, dtype: float64


status,connected,votes_no_edges,total
size_band,,,
1,29691,12529,42220
2-5,12055,1925,13980
6-20,4285,313,4598
21-100,2981,75,3056
>100,2206,26,2232
total,51218,14868,66086


In [6]:
platform_of = dict(zip(nodes["deployment_id"], nodes["platform"]))
pd.crosstab(edges["source"].map(platform_of), edges["target"].map(platform_of))

target,aragon,daohaus,daostack,realms,snapshot,tally
source,,,,,,
aragon,472,265,24,0,3745,279
daohaus,269,5242,67,0,13129,1012
daostack,40,66,78,0,806,92
realms,0,0,0,3556,0,0
snapshot,3,18,0,0,587662,22942
tally,0,0,0,0,54918,5619


In [7]:
multi_voter = with_votes[with_votes["voters_in_votes"] >= 2]

print("Deployments with 2+ voters:", len(multi_voter))
pd.crosstab(
    multi_voter["platform"], multi_voter["status"], normalize="index"
).round(3)

Deployments with 2+ voters: 23866


status,connected,votes_no_edges
platform,,
aragon,0.712,0.288
daohaus,0.917,0.083
daostack,0.788,0.212
realms,0.611,0.389
snapshot,0.918,0.082
tally,0.863,0.137
